# Cat Podcast - Full Pipeline (Audio + Lip Sync Video)
Generates audio with VibeVoice AND animates cats with SadTalker lip sync.

In [ ]:
# Cell 1: Install everything
!apt-get update -qq > /dev/null
!apt-get install -y ffmpeg -qq > /dev/null
!pip install flask pyngrok -qq

import os

# Install VibeVoice
if not os.path.exists('/content/VibeVoice/setup.py'):
    !git clone https://github.com/vibevoice-community/VibeVoice.git > /dev/null 2>&1
%cd /content/VibeVoice
!pip install -e . -qq

# Install SadTalker
if not os.path.exists('/content/SadTalker'):
    %cd /content
    !git clone https://github.com/OpenTalker/SadTalker.git > /dev/null 2>&1
%cd /content/SadTalker
!pip install -r requirements.txt -qq
!mkdir -p checkpoints
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2rc1/motion_exp_generator.pt -O checkpoints/motion_exp_generator.pt
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2rc1/motion_exp_extractor.pt -O checkpoints/motion_exp_extractor.pt
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2rc1/SadTalker_V0.0.2_256.safetensors -O checkpoints/SadTalker_V0.0.2_256.safetensors
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2rc1/SadTalker_V0.0.2_512.safetensors -O checkpoints/SadTalker_V0.0.2_512.safetensors
!wget -q https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc1/facial_analysis_remodeling.pt -O checkpoints/facial_analysis_remodeling.pt

print('Setup done')

In [ ]:
# Cell 2: Server with audio + video generation
import subprocess as sp
sp.run('pkill -f ngrok', shell=True, capture_output=True)
sp.run('rm -rf /tmp/ngrok*', shell=True, capture_output=True)

from flask import Flask, request, jsonify
from pyngrok import ngrok
import subprocess
import base64
import traceback
import os

ngrok.set_auth_token('3Hg6t6KMD50TMDe7WJBy6kxICHC_34gzH8g6o9N4V7We6VDgm')

app = Flask(__name__)

@app.route('/health')
def health():
    return jsonify({'status': 'ok'})

@app.route('/generate', methods=['POST'])
def generate():
    try:
        data = request.json
        script = data.get('script', '')
        filename = data.get('filename', 'episode.txt')
        source_image = data.get('source_image', 'simba')  # 'simba' or 'meow'
        
        # Save script
        with open(filename, 'w') as f:
            f.write(script)
        
        os.makedirs('./outputs', exist_ok=True)
        
        # Step 1: Generate audio with VibeVoice
        cmd = 'python demo/inference_from_file.py --model_path microsoft/VibeVoice-1.5B --txt_path ' + filename + ' --speaker_names Frank Maya --output_dir ./outputs --cfg_scale 1.3 --device cuda'
        subprocess.run(cmd, shell=True, check=True)
        
        # Find audio file
        audio_file = './outputs/' + filename.replace('.txt', '_generated.wav')
        if not os.path.exists(audio_file):
            audio_file = './outputs/episode_script_generated.wav'
        
        # Step 2: Animate with SadTalker
        if source_image == 'simba':
            image_path = '/content/cat_images/simba.jpg'
        else:
            image_path = '/content/cat_images/meow.jpg'
        
        cmd = f'python inference.py --driven_audio {audio_file} --source_image {image_path} --result_dir ./outputs --still --preprocess full --enhancer gfpgan'
        subprocess.run(cmd, shell=True, check=True)
        
        # Find output video
        video_files = [f for f in os.listdir('./outputs') if f.endswith('.mp4')]
        if video_files:
            video_path = os.path.join('./outputs', sorted(video_files)[-1])
            with open(video_path, 'rb') as f:
                video_base64 = base64.b64encode(f.read()).decode('utf-8')
            return jsonify({
                'status': 'success', 
                'video_base64': video_base64,
                'audio_base64': base64.b64encode(open(audio_file, 'rb').read()).decode('utf-8')
            })
        else:
            return jsonify({'status': 'error', 'message': 'No video generated'}), 500
            
    except Exception as e:
        return jsonify({'status': 'error', 'message': str(e), 'trace': traceback.format_exc()}), 500

url = ngrok.connect(5000)
print('\nYOUR WEBHOOK URL: ' + str(url) + '/generate\n')
app.run(port=5000)